In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score


In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/Pavan-gs/ai-services-bbsr/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})
print('Dataset:', X.shape)


Dataset: (7032, 19)


In [3]:
numeric_cols = X.select_dtypes(include='number').columns
categorical_cols = X.select_dtypes(exclude='number').columns
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
candidates = {
    'Decision Tree': (
        DecisionTreeClassifier(random_state=42),
        {
            'model__max_depth': [3, 5, 8, None],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4]
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {
            'model__n_estimators': [100, 200],
            'model__max_depth': [None, 8, 12],
            'model__min_samples_split': [2, 5]
        }
    ),
    'SVM': (
        SVC(probability=True, random_state=42),
        {
            'model__C': [0.1, 1, 10],
            'model__kernel': ['linear', 'rbf'],
            'model__gamma': ['scale', 'auto']
        }
    )
}


In [5]:
searches = {}
tuning_rows = []
for name, (model, grid) in candidates.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    search = GridSearchCV(
        pipe, grid, cv=5, scoring='recall', n_jobs=-1,
        return_train_score=True, refit=True
    )
    search.fit(X_train, y_train)
    searches[name] = search
    i = search.best_index_
    tuning_rows.append({
        'Model': name,
        'Best CV Recall': search.best_score_,
        'Train Recall': search.cv_results_['mean_train_score'][i],
        'Validation Recall': search.cv_results_['mean_test_score'][i],
        'Gap': search.cv_results_['mean_train_score'][i] - search.cv_results_['mean_test_score'][i],
        'Best Parameters': search.best_params_
    })
tuning_df = pd.DataFrame(tuning_rows)
tuning_df


,Model,Best CV Recall,Train Recall,Validation Recall,Gap,Best Parameters
0,Decision Tree,0.542475,0.556355,0.542475,0.013880,"{'model__max_depth': 5, 'model__min_samples_le..."
1,Random Forest,0.522408,0.912040,0.522408,0.389632,"{'model__max_depth': 12, 'model__min_samples_s..."
2,SVM,0.533779,0.542475,0.533779,0.008696,"{'model__C': 10, 'model__gamma': 'scale', 'mod..."


In [6]:
test_rows = []
for name, search in searches.items():
    best = search.best_estimator_
    pred = best.predict(X_test)
    prob = best.predict_proba(X_test)[:, 1]
    test_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })
test_results = pd.DataFrame(test_rows)
test_results.round(4)


,Model,Accuracy,Precision,Recall,ROC-AUC
0,Decision Tree,0.7896,0.6021,0.6150,0.8289
1,Random Forest,0.7910,0.6351,0.5027,0.8237
2,SVM,0.7974,0.6378,0.5508,0.8271


In [7]:
tuning_df[['Model', 'Train Recall', 'Validation Recall', 'Gap']].round(4)


,Model,Train Recall,Validation Recall,Gap
0,Decision Tree,0.5564,0.5425,0.0139
1,Random Forest,0.9120,0.5224,0.3896
2,SVM,0.5425,0.5338,0.0087


In [8]:
learning_curves = {}
for name, search in searches.items():
    sizes, train_scores, val_scores = learning_curve(
        search.best_estimator_, X_train, y_train,
        cv=5, scoring='recall',
        train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0],
        n_jobs=-1
    )
    curve = pd.DataFrame({
        'Training Size': sizes,
        'Train Recall': train_scores.mean(axis=1),
        'Validation Recall': val_scores.mean(axis=1)
    })
    learning_curves[name] = curve
    print('\n' + name)
    print(curve.round(4).to_string(index=False))



Decision Tree
 Training Size  Train Recall  Validation Recall
           900        0.6025             0.4441
          1800        0.5939             0.5278
          2700        0.5539             0.5184
          3600        0.5488             0.5231
          4500        0.5564             0.5425

Random Forest
 Training Size  Train Recall  Validation Recall
           900        0.9955             0.4502
          1800        0.9849             0.5177
          2700        0.9658             0.5137
          3600        0.9360             0.5237
          4500        0.9120             0.5224

SVM
 Training Size  Train Recall  Validation Recall
           900        0.5240             0.4816
          1800        0.5442             0.5144
          2700        0.5421             0.5311
          3600        0.5324             0.5298
          4500        0.5425             0.5338


In [9]:
champion_name = tuning_df.sort_values('Best CV Recall', ascending=False).iloc[0]['Model']
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_
joblib.dump(champion_model, '../models/churn_v3.joblib')
print('Champion:', champion_name)
print('Best parameters:', champion_search.best_params_)
print('Best CV recall:', round(champion_search.best_score_, 4))
print('Saved: ../models/churn_v3.joblib')


Champion: Decision Tree
Best parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}
Best CV recall: 0.5425
Saved: ../models/churn_v3.joblib


In [10]:
print('DAY 3 - TUNING SUMMARY')
print('-' * 60)
for _, row in tuning_df.iterrows():
    print(f"{row['Model']}: CV Recall={row['Best CV Recall']:.4f}, Train Recall={row['Train Recall']:.4f}, Gap={row['Gap']:.4f}")
    print('  Parameters:', row['Best Parameters'])
print('\nTuned champion:', champion_name)
print('Best CV recall:', f'{champion_search.best_score_:.4f}')
print('Artifact: models/churn_v3.joblib')


DAY 3 - TUNING SUMMARY
------------------------------------------------------------
Decision Tree: CV Recall=0.5425, Train Recall=0.5564, Gap=0.0139
  Parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}
Random Forest: CV Recall=0.5224, Train Recall=0.9120, Gap=0.3896
  Parameters: {'model__max_depth': 12, 'model__min_samples_split': 2, 'model__n_estimators': 100}
SVM: CV Recall=0.5338, Train Recall=0.5425, Gap=0.0087
  Parameters: {'model__C': 10, 'model__gamma': 'scale', 'model__kernel': 'linear'}

Tuned champion: Decision Tree
Best CV recall: 0.5425
Artifact: models/churn_v3.joblib
